# Train a drug–disease link prediction GNN

Run beside the existing `kg_data` folder or edit the path in the first Python cell. If imports are missing, run `%pip install torch scikit-learn pandas` in a separate cell, then restart the kernel if needed.

This baseline learns from PrimeKG indications for diseases with gene neighbors and drugs with known targets. Entire diseases are held out for validation and testing. Indication, contraindication, and off-label edges are excluded from message passing. ECoGraph subtypes receive scores at inference time using the same gene-to-disease aggregation as PrimeKG diseases. This is a transfer assumption: observed mutations provide gene context for the subtype. The two source relations remain distinct in the exported graph.

The held-out test measures recovery of PrimeKG indications against sampled unlabeled drug pairs. The four ECoGraph subtype rankings require separate validation.

In [1]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score
from torch import nn
from torch.nn import functional as F

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "agent/config.py").is_file() and (p / "src/primekg").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter within the BioHack2026 checkout.")

graph_dir = project_root / "data/processed/primekg_ecograph"
results_dir = project_root / "artifacts/predictions/colorectal"
results_dir.mkdir(parents=True, exist_ok=True)
seed = 42
rng = np.random.default_rng(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

nodes = pd.read_csv(graph_dir / "nodes.csv", dtype={"node_id": str}).set_index("node_index")
assert nodes.index.is_unique

mutation_relation = "ecograph_subtype_has_observed_mutation_in_gene"
context_relations = ["protein_protein", "drug_protein", "disease_protein", mutation_relation]
label_relations = ["indication", "off-label use", "contraindication"]
keep = context_relations + label_relations

edges = pd.concat(
    (chunk.loc[chunk["relation"].isin(keep)] for chunk in pd.read_csv(
        graph_dir / "edges.csv", usecols=["x_index", "y_index", "relation"],
        chunksize=250_000,
    )),
    ignore_index=True,
)
assert set(edges["x_index"]) | set(edges["y_index"]) <= set(nodes.index)

def pairs(relations, left_type, right_type):
    """Return unique pairs in type order, including when the CSV stores both directions."""
    rows = edges.loc[edges["relation"].isin(relations)]
    forward = rows["x_index"].map(nodes["node_type"]).eq(left_type)
    left = rows["x_index"].where(forward, rows["y_index"])
    right = rows["y_index"].where(forward, rows["x_index"])
    assert left.map(nodes["node_type"]).eq(left_type).all()
    assert right.map(nodes["node_type"]).eq(right_type).all()
    return pd.DataFrame({"left": left, "right": right}).drop_duplicates()

ppi = pairs(["protein_protein"], "gene/protein", "gene/protein")
targets = pairs(["drug_protein"], "drug", "gene/protein")
disease_genes = pairs(["disease_protein", mutation_relation], "disease", "gene/protein")
known = pairs(["indication"], "disease", "drug")
excluded_pairs = pairs(label_relations, "disease", "drug")

genes = pd.Index(nodes.index[nodes["node_type"].eq("gene/protein")])
drugs = pd.Index(sorted(targets["left"].unique()))
diseases = pd.Index(sorted(disease_genes["left"].unique()))
assert len(genes) and len(drugs) and len(diseases)
positives = known.loc[known["left"].isin(diseases) & known["right"].isin(drugs)].copy()
print(f"{len(positives):,} of {len(known):,} known indication pairs have both kinds of gene context.")

/var/folders/_y/7_8r9bls3fs6c2g_xfpj0m3r0000gn/T/ipykernel_8922/2975300431.py:17: DtypeWarning: Columns (6: ecograph_subtype, 7: node_kind) have mixed types. Specify dtype option on import or set low_memory=False.
  nodes = pd.read_csv(graph_dir / "nodes.csv", dtype={"node_id": str}).set_index("node_index")


6,233 of 9,388 known indication pairs have both kinds of gene context.


In [2]:
# Split by disease before training. Reverse copies of each edge were deduplicated.
eligible = rng.permutation(positives["left"].unique())
assert len(eligible) >= 10, "Too few eligible diseases for this split"
val_size = max(1, round(0.15 * len(eligible)))
val_ids = eligible[:val_size]
test_ids = eligible[val_size:2 * val_size]
train_ids = eligible[2 * val_size:]

def labeled_positive_pairs(ids):
    rows = positives.loc[positives["left"].isin(ids)]
    return np.column_stack([diseases.get_indexer(rows["left"]), drugs.get_indexer(rows["right"])])

train_pos, val_pos, test_pos = map(labeled_positive_pairs, [train_ids, val_ids, test_ids])
for name, ids, split in zip(
    ["train", "validation", "test"], [train_ids, val_ids, test_ids], [train_pos, val_pos, test_pos]
):
    print(f"{name:10s} {len(ids):5,} diseases, {len(split):6,} known indications")

# Sample one unlabeled drug per positive, matched on disease.
# Exclude all recorded indications, off-label uses, and contraindications.
forbidden = {
    disease: set(drugs.get_indexer(group["right"])) - {-1}
    for disease, group in excluded_pairs.groupby("left")
}
negative_pools = {}
for disease in eligible:
    allowed = np.array(sorted(set(range(len(drugs))) - forbidden.get(disease, set())), dtype=int)
    negative_pools[diseases.get_loc(disease)] = allowed

def sample_negatives(positive_pairs):
    sampled = []
    for disease, n_pairs in zip(*np.unique(positive_pairs[:, 0], return_counts=True)):
        allowed = negative_pools[disease]
        if len(allowed) < n_pairs:
            raise ValueError(f"Too few unlabeled drugs for disease {diseases[disease]}")
        drug_ids = rng.choice(allowed, n_pairs, replace=False)
        sampled.extend((disease, drug) for drug in drug_ids)
    return np.asarray(sampled, dtype=np.int64)

val_neg = sample_negatives(val_pos)
test_neg = sample_negatives(test_pos)

train        633 diseases,  4,229 known indications
validation   136 diseases,  1,151 known indications
test         136 diseases,    853 known indications


In [3]:
# Create sparse matrices that average the gene messages received by each node.
def mean_adjacency(rows, receiver_ids, sender_ids):
    receiver = receiver_ids.get_indexer(rows["left"])
    sender = sender_ids.get_indexer(rows["right"])
    assert (receiver >= 0).all() and (sender >= 0).all()
    degree = np.bincount(receiver, minlength=len(receiver_ids))
    indices = torch.tensor(np.stack([receiver, sender]), dtype=torch.long, device=device)
    weights = torch.tensor(1.0 / degree[receiver], dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(
        indices, weights, (len(receiver_ids), len(sender_ids))
    ).coalesce()

ppi_both = pd.concat([ppi, ppi.rename(columns={"left": "right", "right": "left"})])
ppi_both = ppi_both.drop_duplicates()
adj_ppi = mean_adjacency(ppi_both, genes, genes)
adj_drug = mean_adjacency(targets, drugs, genes)
adj_disease = mean_adjacency(disease_genes, diseases, genes)

class IndicationGNN(nn.Module):
    def __init__(self, width=48):
        super().__init__()
        self.gene = nn.Embedding(len(genes), width)
        self.gene_self = nn.Linear(width, width)
        self.gene_ppi = nn.Linear(width, width, bias=False)
        self.drug_projection = nn.Linear(width, width)
        self.disease_projection = nn.Linear(width, width)
        self.decoder = nn.Sequential(nn.Linear(3 * width, width), nn.ReLU(), nn.Linear(width, 1))

    def encode(self):
        initial = self.gene.weight
        neighbors = torch.sparse.mm(adj_ppi, initial)
        gene_vectors = F.relu(self.gene_self(initial) + self.gene_ppi(neighbors))
        drug_vectors = self.drug_projection(torch.sparse.mm(adj_drug, gene_vectors))
        disease_vectors = self.disease_projection(torch.sparse.mm(adj_disease, gene_vectors))
        return disease_vectors, drug_vectors

    def score(self, encoded, pairs):
        ids = torch.as_tensor(pairs, dtype=torch.long, device=device)
        disease = encoded[0][ids[:, 0]]
        drug = encoded[1][ids[:, 1]]
        return self.decoder(torch.cat([disease, drug, disease * drug], dim=1)).squeeze(1)

model = IndicationGNN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.001)

def examples(positive, negative):
    return np.concatenate([positive, negative]), np.r_[np.ones(len(positive)), np.zeros(len(negative))]

@torch.no_grad()
def evaluate(positive, negative):
    model.eval()
    pairs, labels = examples(positive, negative)
    scores = model.score(model.encode(), pairs).cpu().numpy()
    return average_precision_score(labels, scores)

best_ap = -1.0
for epoch in range(1, 41):
    model.train()
    optimizer.zero_grad()
    batch, labels = examples(train_pos, sample_negatives(train_pos))
    logits = model.score(model.encode(), batch)
    loss = F.binary_cross_entropy_with_logits(
        logits, torch.as_tensor(labels, dtype=torch.float32, device=device)
    )
    loss.backward()
    optimizer.step()

    val_ap = evaluate(val_pos, val_neg)
    if val_ap > best_ap:
        best_ap, best_epoch = val_ap, epoch
        best_weights = deepcopy(model.state_dict())
    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | loss {loss.item():.3f} | validation AP {val_ap:.3f}")

model.load_state_dict(best_weights)
test_ap = evaluate(test_pos, test_neg)
test_pairs, test_labels = examples(test_pos, test_neg)
drug_popularity = np.bincount(train_pos[:, 1], minlength=len(drugs))
baseline_ap = average_precision_score(test_labels, drug_popularity[test_pairs[:, 1]])
print(f"Best epoch {best_epoch} | validation AP {best_ap:.3f}")
print(f"Held-out disease test AP {test_ap:.3f} | drug popularity baseline {baseline_ap:.3f}")

/var/folders/_y/7_8r9bls3fs6c2g_xfpj0m3r0000gn/T/ipykernel_8922/4079294675.py:9: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:923.)
  return torch.sparse_coo_tensor(


Epoch 01 | loss 0.694 | validation AP 0.602
Epoch 05 | loss 0.681 | validation AP 0.678
Epoch 10 | loss 0.635 | validation AP 0.689
Epoch 15 | loss 0.556 | validation AP 0.712
Epoch 20 | loss 0.513 | validation AP 0.757
Epoch 25 | loss 0.487 | validation AP 0.780
Epoch 30 | loss 0.452 | validation AP 0.794
Epoch 35 | loss 0.426 | validation AP 0.804
Epoch 40 | loss 0.417 | validation AP 0.813
Best epoch 40 | validation AP 0.813
Held-out disease test AP 0.819 | drug popularity baseline 0.894


In [4]:
# Rank drugs with known targets for each ECoGraph subtype.
subtypes = nodes.loc[nodes["node_kind"].eq("molecular_subtype"), "ecograph_subtype"]
assert len(subtypes) == 4 and subtypes.index.isin(diseases).all()

model.eval()
rankings = []
with torch.no_grad():
    encoded = model.encode()
    for node_id, subtype in subtypes.items():
        disease_position = diseases.get_loc(node_id)
        pairs_to_rank = np.column_stack([
            np.full(len(drugs), disease_position), np.arange(len(drugs))
        ])
        scores = model.score(encoded, pairs_to_rank).cpu().numpy()
        rows = nodes.loc[drugs, ["node_id", "node_name"]].reset_index()
        rows = rows.rename(columns={
            "node_index": "drug_node_index", "node_id": "drug_id", "node_name": "drug"
        })
        rows["subtype"] = subtype
        rows["ranking_score"] = scores
        rankings.append(rows.sort_values("ranking_score", ascending=False).head(20))

rankings = pd.concat(rankings, ignore_index=True)
rankings["rank"] = rankings.groupby("subtype").cumcount() + 1
rankings.to_csv(results_dir / "subtype_drug_rankings.csv", index=False)
display(rankings[["subtype", "rank", "drug", "drug_id", "ranking_score"]])

,subtype,rank,drug,drug_id,ranking_score
0,subtype:GI.CIN,1,Ulobetasol,DB00596,3.189067
1,subtype:GI.CIN,2,Desoximetasone,DB00547,2.927836
2,subtype:GI.CIN,3,ORG-34517,DB05423,2.927836
3,subtype:GI.CIN,4,Clocortolone,DB00838,2.927836
4,subtype:GI.CIN,5,Diflorasone,DB00223,2.927836
...,...,...,...,...,...
75,subtype:GI.HM-indel,16,Flurandrenolide,DB00846,2.329333
76,subtype:GI.HM-indel,17,Flumethasone,DB00663,2.329333
77,subtype:GI.HM-indel,18,Flunisolide,DB00180,2.329333
78,subtype:GI.HM-indel,19,Alclometasone,DB00240,2.329333


In [5]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(rankings[["subtype", "rank", "drug", "drug_id", "ranking_score"]])

,subtype,rank,drug,drug_id,ranking_score
0,subtype:GI.CIN,1,Ulobetasol,DB00596,3.189067
1,subtype:GI.CIN,2,Desoximetasone,DB00547,2.927836
2,subtype:GI.CIN,3,ORG-34517,DB05423,2.927836
3,subtype:GI.CIN,4,Clocortolone,DB00838,2.927836
4,subtype:GI.CIN,5,Diflorasone,DB00223,2.927836
5,subtype:GI.CIN,6,Neramexane,DB04926,2.727356
6,subtype:GI.CIN,7,Paramethasone,DB01384,2.517122
7,subtype:GI.CIN,8,Fluorometholone,DB00324,2.517122
8,subtype:GI.CIN,9,Rimexolone,DB00896,2.517122
9,subtype:GI.CIN,10,Ambroxol acefyllinate,DB13141,2.505425


Average precision (AP) uses one sampled unlabeled pair per known indication. Compare the GNN to the training drug-popularity baseline on exactly the same test pairs. The subtype output contains raw ranking scores for exploration. Cohort size, mutation burden, and differences between PrimeKG disease–gene associations and ECoGraph mutation observations can affect how well the model transfers.

[PyTorch sparse matrix multiplication](https://docs.pytorch.org/docs/stable/generated/torch.sparse.mm) · [Average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)